[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/%E6%95%B8%E6%93%9A%E6%95%B4%E7%90%86_%E9%80%A3%E6%8E%A5_%E5%90%88%E4%BD%B5_%E9%87%8D%E5%A1%91/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E7%AF%84%E4%BE%8B_%E8%AA%B0%E6%9C%80%E6%9C%83%E7%94%A8%E9%9B%BB_%E4%BA%BA%E5%8F%A3%E8%88%87%E7%94%A8%E9%9B%BB%E8%B3%87%E6%96%99%E5%A4%A7%E5%90%88%E4%BD%B5.ipynb)

# 🔗 誰最會用電？把人口與用電資料合併成一張表

> 🎯 **情境**：新聞說「某某縣市最耗電」，但人多的地方本來就用比較多電，這樣比公平嗎？要回答「**每個人**用多少電」，需要兩個機關的資料：台電的**各縣市用電**，和內政部的**各縣市人口**。今天你要把三年的人口檔接起來（concat）、再和用電資料合併（merge）——然後你會發現，**兩個機關連「臺北市」怎麼寫都不一樣**。

| 素材檔案 | 內容 |
| --- | --- |
| `各鄉鎮市區人口密度_112年.json` | 內政部各鄉鎮市區年底人口與面積：民國 112 年（2023） |
| `各鄉鎮市區人口密度_113年.json` | 同上：民國 113 年（2024） |
| `各鄉鎮市區人口密度_114年.json` | 同上：民國 114 年（2025） |
| `台電各縣市用電統計.csv` | 台電各縣市每月售電量（住宅、服務業、工業、農林漁牧） |
| `臺北市各區資源回收量.csv` | 臺北市各區每月資源回收量（Big5、民國年；練習題用） |

| 📡 資料來源（政府資料開放平台） | 提供機關 | 下載日期 |
| --- | --- | --- |
| [各鄉鎮市區人口密度（民國 112 年底）](https://data.gov.tw/dataset/8410) | 內政部 | 2026-10-24 |
| [各鄉鎮市區人口密度（民國 113 年底）](https://data.gov.tw/dataset/8410) | 內政部 | 2026-10-24 |
| [各鄉鎮市區人口密度（民國 114 年底）](https://data.gov.tw/dataset/8410) | 內政部 | 2026-10-24 |
| [台灣電力公司各縣市住宅、服務業及機關用電統計資料](https://data.gov.tw/dataset/29935) | 台灣電力股份有限公司 | 2026-10-24 |
| [臺北市各區清潔隊資源回收量](https://data.gov.tw/dataset/121705) | 臺北市政府環境保護局 | 2026-10-24 |

> ✅ 這些都是**真實的政府開放資料**，依「政府資料開放授權條款－第1版」使用。課堂上統一使用上表日期的快照，所以每個人跑出來的數字都一樣。

## 🗺️ 今日任務地圖

| 關卡 | 要回答的問題 |
| --- | --- |
| ⬜ 任務 1 | 三年的人口檔，上下接成一張表——`concat` |
| ⬜ 任務 2 | 數字是文字、還有「…」——轉型態 |
| ⬜ 任務 3 | 把用電和人口合併——`merge`，然後……對不上！ |
| ⬜ 任務 4 | 寬表與長表——`pivot` 和 `melt` |
| ⬜ 任務 5 | 哪些縣市人口在增加？——長表轉寬表算成長率 |

**怎麼用這本 Notebook**：從上到下按 `Shift + Enter` 一格一格執行。每個任務都會先講「你想知道什麼」，再給程式，最後用 💡 **看懂結果** 告訴你這個數字代表什麼。

In [1]:
# 📦 準備素材：在 Colab 開啟時，自動下載本章素材壓縮檔並解壓縮
#    （若是在自己電腦解壓縮後開啟，檔案已經在旁邊，會直接使用）
import os, zipfile, urllib.request

FILES = [
    '各鄉鎮市區人口密度_112年.json',
    '各鄉鎮市區人口密度_113年.json',
    '各鄉鎮市區人口密度_114年.json',
    '台電各縣市用電統計.csv',
    '臺北市各區資源回收量.csv',
]
ZIP_URL = 'https://github.com/roberthsu2003/PythonForDataAnalysis/raw/master/%E6%95%B8%E6%93%9A%E6%95%B4%E7%90%86_%E9%80%A3%E6%8E%A5_%E5%90%88%E4%BD%B5_%E9%87%8D%E5%A1%91/%E6%95%B8%E6%93%9A%E6%95%B4%E7%90%86_%E9%80%A3%E6%8E%A5_%E5%90%88%E4%BD%B5_%E9%87%8D%E5%A1%91_%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90.zip'

if not all(os.path.exists(f) for f in FILES):
    urllib.request.urlretrieve(ZIP_URL, 'materials.zip')            # 下載本章素材壓縮檔
    with zipfile.ZipFile('materials.zip') as z:
        for name in z.namelist():
            inner = name.split('/', 1)[1] if '/' in name else name   # 去掉壓縮檔最外層的資料夾名稱
            if inner and not name.endswith('/'):
                os.makedirs(os.path.dirname(inner) or '.', exist_ok=True)
                with z.open(name) as src, open(inner, 'wb') as dst:
                    dst.write(src.read())

print('✅ 素材準備完成：', [f for f in FILES if os.path.exists(f)])

✅ 素材準備完成： ['各鄉鎮市區人口密度_112年.json', '各鄉鎮市區人口密度_113年.json', '各鄉鎮市區人口密度_114年.json', '台電各縣市用電統計.csv', '臺北市各區資源回收量.csv']


## 任務 1：三年的人口檔，上下接成一張表——`concat`

**白話說明**：三個 JSON 檔欄位一模一樣，只是年份不同，用 `pd.concat` 上下接起來。內政部的 API 把資料放在 `responseData` 這一層。

In [2]:
import json
import pandas as pd

frames = []
for year in [112, 113, 114]:
    with open(f'各鄉鎮市區人口密度_{year}年.json', encoding='utf-8') as f:
        frames.append(pd.DataFrame(json.load(f)['responseData']))     # 每一年一張表

pop = pd.concat(frames, ignore_index=True)                            # 上下接起來，重新編列號
pop.columns = ['年度', '區域', '人口', '面積', '人口密度']
print(pop.shape)
print(pop.dtypes)                                                     # 全部都是文字！
pop.head()

(1110, 5)
年度      str
區域      str
人口      str
面積      str
人口密度    str
dtype: object


,年度,區域,人口,面積,人口密度
0,112,新北市板橋區,554247,23.1373,23955
1,112,新北市三重區,383108,16.3170,23479
2,112,新北市中和區,407273,20.1440,20218
3,112,新北市永和區,214771,5.7138,37588
4,112,新北市新莊區,423607,19.7383,21461


## 任務 2：數字是文字、還有「…」——轉型態

**白話說明**：API 回傳的數字全部是**文字**，要用 `pd.to_numeric` 轉成數字。但轉換時會出錯——因為**東沙群島、南沙群島的人口寫成「…」**（沒有常住人口統計）。`errors='coerce'` 讓轉不了的變成 `NaN`。

In [3]:
print(pop[~pop['人口'].str.isdigit()])                               # 哪些「人口」不是純數字？

pop['人口'] = pd.to_numeric(pop['人口'], errors='coerce')             # 「…」→ NaN
pop['縣市'] = pop['區域'].str[:3]                                      # 「新北市板橋區」→「新北市」
pop['西元年'] = pop['年度'].astype(int) + 1911

total = pop.groupby('西元年')['人口'].sum()
display(total.to_frame('全台人口').style.format('{:,.0f}'))
print('兩年少了：', f'{total[2023] - total[2025]:,.0f} 人')

       年度    區域 人口      面積 人口密度
368   112  東沙群島  …  2.3800    …
369   112  南沙群島  …  0.5045    …
738   113  東沙群島  …  2.3800    …
739   113  南沙群島  …  0.5045    …
1108  114  東沙群島  …    2.38    …
1109  114  南沙群島  …  0.5045    …


,全台人口
西元年,
2023,"23,420,442"
2024,"23,400,220"
2025,"23,299,132"


兩年少了： 121,310 人


> 💡 **看懂結果**：台灣人口從 2023 年底的 2,342 萬，到 2025 年底剩 2,330 萬——**兩年少了 12 萬人**，差不多是整個澎湖縣的人口不見了。少子化正在發生。

## 任務 3：把用電和人口合併——`merge`，然後……對不上！

**白話說明**：`merge` 依照「共同的欄位」把兩張表左右接起來，就像 Excel 的 VLOOKUP。加上 `indicator=True` 會多一欄 `_merge`，告訴你每一列是「兩邊都有」還是「只有一邊有」——**合併後一定要檢查**。

In [4]:
pd.set_option('display.float_format', '{:,.0f}'.format)                 # 大數字加千分位、不用科學記號（5.67e+08 很難讀）

power = pd.read_csv('台電各縣市用電統計.csv', thousands=',', usecols=range(5)).dropna(subset=['年月'])
home2025 = (power[(power['年月'] >= 11401) & (power['年月'] <= 11412) & (power['用電性質'] == '住宅')]
            .groupby('縣市', as_index=False)['售電量(度)'].sum())                 # 2025 年各縣市住宅用電

pop2025 = pop[pop['西元年'] == 2025].groupby('縣市', as_index=False)['人口'].sum()

merged = home2025.merge(pop2025, on='縣市', how='outer', indicator=True)
print(merged['_merge'].value_counts())
merged[merged['_merge'] != 'both'].sort_values('縣市')

_merge
both          18
right_only     6
left_only      4
Name: count, dtype: int64


,縣市,售電量(度),人口,_merge
1,南沙群,NaN,0,right_only
2,台中市,"6,832,912,297",NaN,left_only
3,台北市,"5,642,341,948",NaN,left_only
4,台南市,"4,423,929,876",NaN,left_only
5,台東縣,"567,143,918",NaN,left_only
15,東沙群,NaN,0,right_only
18,臺中市,NaN,"2,868,465",right_only
19,臺北市,NaN,"2,439,507",right_only
20,臺南市,NaN,"1,852,477",right_only
21,臺東縣,NaN,"208,320",right_only


> 😱 **只有 18 個縣市對得上！** 仔細看：台電寫「**台**北市、**台**中市、**台**南市、**台**東縣」，內政部寫「**臺**北市、**臺**中市……」。
> 對電腦來說，「台」和「臺」是兩個完全不同的字。另外「東沙群」「南沙群」是人口檔切前三個字時切出來的，它們本來就沒有用電資料。
>
> **這就是真實世界的資料合併**：每個機關有自己的寫法，合併前一定要先統一「鑰匙」。

In [5]:
home2025['縣市'] = home2025['縣市'].str.replace('台', '臺')            # 統一寫法：台 → 臺

merged = home2025.merge(pop2025, on='縣市', how='inner', validate='one_to_one')   # validate：確認兩邊的縣市都不重複
print('合併後的縣市數：', len(merged))
merged['每人住宅用電(度)'] = (merged['售電量(度)'] / merged['人口']).round(0)
merged.sort_values('每人住宅用電(度)', ascending=False)

合併後的縣市數： 22


,縣市,售電量(度),人口,每人住宅用電(度)
4,臺東縣,"567,143,918","208,320","2,722"
18,連江縣,"35,286,687","13,621","2,591"
12,新竹市,"1,175,100,980","455,740","2,578"
13,新竹縣,"1,526,538,475","597,235","2,556"
14,桃園市,"5,700,761,727","2,355,106","2,421"
8,宜蘭縣,"1,077,814,524","449,336","2,399"
3,臺南市,"4,423,929,876","1,852,477","2,388"
1,臺中市,"6,832,912,297","2,868,465","2,382"
11,新北市,"9,367,871,747","4,044,831","2,316"
2,臺北市,"5,642,341,948","2,439,507","2,313"


> 💡 **看懂結果**：每人住宅用電最多的是**臺東縣、連江縣、新竹市、新竹縣**；最少的是**金門縣**。南部縣市不一定比較高——用電還跟家裡的人數、房子大小、電器有關。
>
> 🤖 **換你問 AI**：「我要用『縣市』合併兩張表，但一張寫『台北市』、一張寫『臺北市』。請先統一寫法再合併，並用 indicator=True 檢查有沒有對不到的資料。」

## 任務 4：寬表與長表——`pivot` 和 `melt`

**白話說明**：
- **長表**（一列一個「縣市＋用電類別」）適合存資料、畫圖；
- **寬表**（一列一個縣市，用電類別攤成欄位）適合人看、做比較。

`pivot_table` 把長表變寬表，`melt` 把寬表變回長表。

In [6]:
y2025 = power[(power['年月'] >= 11401) & (power['年月'] <= 11412)].copy()
y2025['縣市'] = y2025['縣市'].str.replace('台', '臺')

wide = y2025.pivot_table(index='縣市', columns='用電性質', values='售電量(度)', aggfunc='sum')   # 長表 → 寬表
wide = wide.join(pop2025.set_index('縣市'))                                               # join：依索引合併
per_person = wide[['住宅', '服務業', '工業', '農林漁牧']].div(wide['人口'], axis=0).round(0)  # 每一欄都除以人口
per_person['合計'] = per_person.sum(axis=1)
per_person.sort_values('合計', ascending=False).head(6)

,住宅,服務業,工業,農林漁牧,合計
縣市,,,,,
新竹縣,"2,556","1,781","16,346",39,"20,722"
新竹市,"2,578","2,613","14,633",21,"19,845"
臺南市,"2,388","1,710","13,482",244,"17,824"
苗栗縣,"2,260","1,288","11,007",65,"14,620"
桃園市,"2,421","1,971","8,005",26,"12,423"
臺中市,"2,382","1,997","7,485",55,"11,919"


> 💡 **看懂結果**：算上工業用電，第一名是**新竹縣**——每人一年 2 萬度電，是全台平均（約 9,900 度）的 2 倍多！因為新竹科學園區的晶圓廠就在這裡；第三名臺南市則有南部科學園區。
> **同樣是「每人用電」，只看住宅和看全部，答案完全不同。** 報告時一定要說清楚你算的是哪一種。

In [7]:
long = per_person.drop(columns='合計').reset_index().melt(id_vars='縣市', var_name='用電性質', value_name='每人度數')   # 寬表 → 長表
print(long.shape)
long.head(8)

(88, 3)


,縣市,用電性質,每人度數
0,南投縣,住宅,"2,065"
1,嘉義市,住宅,"2,227"
2,嘉義縣,住宅,"2,100"
3,基隆市,住宅,"2,185"
4,宜蘭縣,住宅,"2,399"
5,屏東縣,住宅,"2,231"
6,彰化縣,住宅,"2,178"
7,新北市,住宅,"2,316"


## 任務 5：哪些縣市人口在增加？——長表轉寬表算成長率

In [8]:
county = pop.pivot_table(index='縣市', columns='西元年', values='人口', aggfunc='sum').dropna()   # 縣市 × 年
county['兩年變化%'] = ((county[2025] / county[2023] - 1) * 100).round(2)
county.sort_values('兩年變化%', ascending=False)

西元年,2023,2024,2025,兩年變化%
縣市,,,,
桃園市,"2,317,445","2,338,648","2,355,106",2
新竹縣,"589,289","594,641","597,235",1
臺中市,"2,845,909","2,860,601","2,868,465",1
新北市,"4,041,120","4,047,001","4,044,831",0
宜蘭縣,"449,890","449,212","449,336",-0
新竹市,"456,475","457,242","455,740",-0
臺南市,"1,859,946","1,858,651","1,852,477",-0
苗栗縣,"534,575","532,854","531,172",-1
基隆市,"362,255","361,441","359,836",-1


> 💡 **看懂結果**：只有**桃園市、新竹縣、臺中市、新北市**人口在增加（桃園 +1.6%），其他縣市都在減少，**臺北市兩年少了將近 3%**。
> 房價、工作機會、交通——人口會往「住得起、又有工作」的地方移動。

<details><summary>🔄 想看最新資料？點我</summary>

資料每天、每月都在更新。想用今天的數字重跑一次，可以把這段程式貼到新的一格執行：

```python
import pandas as pd, urllib.request, json
year = 115                                                   # 改成想查的民國年（每年年中後才會有前一年的資料）
raw = urllib.request.urlopen(f'https://www.ris.gov.tw/rs-opendata/api/v1/datastore/ODRP048/{year}').read()
pd.DataFrame(json.loads(raw)['responseData']).head()
```

> ⚠️ 開放資料的欄位偶爾會改名或調整，如果最新資料跑出錯誤，正好練習看懂錯誤訊息、對照欄位名稱修正程式。

</details>

---

## ✍️ 換你動手：練習區

每一題都有三格：**① 半成品程式**（把 `___` 換成正確的程式）→ **② ✅ 檢查格**（自動告訴你對不對）→ **③ 🔑 參考解答**（折疊起來，先自己試）。

> 看到 `NameError: name '___' is not defined` 不用緊張，那代表還有空格沒填。

In [9]:
# @title 🧰 練習檢查小幫手（執行一次就好，不用看懂裡面的程式）
import math
import numpy as np
import pandas as pd


def _same(ans, expected, ordered=False):
    if isinstance(expected, (pd.DataFrame, pd.Series)):
        if type(ans) is not type(expected):
            return False
        try:
            assert_equal = pd.testing.assert_frame_equal if isinstance(expected, pd.DataFrame) else pd.testing.assert_series_equal
            if not ordered:
                ans, expected = ans.sort_index(), expected.sort_index()
            assert_equal(ans, expected, check_names=False, check_dtype=False,
                         check_index_type=False, check_exact=False, rtol=1e-3, atol=0.06,
                         **({'check_column_type': False} if isinstance(expected, pd.DataFrame) else {}))
            return True
        except (AssertionError, TypeError, ValueError):
            return False
    if isinstance(expected, np.ndarray):
        return np.asarray(ans).shape == expected.shape and np.allclose(np.asarray(ans, dtype=float), expected, atol=0.06)
    if isinstance(expected, (int, float, np.number)) and not isinstance(expected, bool):
        try:
            return math.isclose(float(ans), float(expected), rel_tol=1e-3, abs_tol=0.06)
        except (TypeError, ValueError):
            return False
    if isinstance(expected, (list, tuple, set)):
        return list(ans) == list(expected) if not isinstance(expected, set) else set(ans) == expected
    return ans == expected


def check(name, expected, ordered=False):
    """比對學生的變數 name 和正確答案 expected，印出 ✅ 或 ❌；ordered=True 時連排列順序也要一樣。"""
    if name not in globals():
        print(f'⏳ 還找不到變數 {name}：請先把上一格的 ___ 填好並執行')
        return
    ans = globals()[name]
    if _same(ans, expected, ordered):
        print(f'✅ {name} 答對了！')
        return
    print(f'❌ {name} 還不對，再檢查看看：')
    if type(ans) is not type(expected):
        print(f'   型態不同：你的是 {type(ans).__name__}，應該是 {type(expected).__name__}')
    elif hasattr(expected, 'shape') and getattr(ans, 'shape', None) != expected.shape:
        print(f'   大小不同：你的是 {ans.shape}，應該是 {expected.shape}')
    else:
        print('   型態和大小都對了，是數值' + ('或排列順序' if ordered else '') + '不一樣——檢查一下篩選條件或計算方式')


print('🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格')

🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格


### ✍️ 練習 1：2025 年底，全台灣有多少人？

<details><summary>💡 卡住了？點我看提示</summary>

任務 2 已經算好 `total`（依西元年加總的人口），用 `total[2025]` 取出來。

</details>

In [ ]:
# ✏️ 練習 1：把 ___ 換成正確的程式，再按執行
people_2025 = total[___]
print(f'2025 年底全台人口：{people_2025:,.0f} 人')

In [ ]:
# @title ✅ 執行這格，檢查練習 1
check('people_2025', pop.loc[pop['西元年'] == 2025, '人口'].sum())

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
people_2025 = total[2025]
print(f'2025 年底全台人口：{people_2025:,.0f} 人')
```

</details>

### ✍️ 練習 2：臺北市哪一區「每人」資源回收最多？

<details><summary>💡 卡住了？點我看提示</summary>

回收資料的「區隊」寫成 `大安區隊`，人口資料的「區域」寫成 `臺北市大安區`。把區隊改成同樣的寫法：`'臺北市' + 區隊.str.replace('區隊', '區')`，再合併。民國 114 年就是 2025 年。

</details>

In [ ]:
# ✏️ 練習 2：把 ___ 換成正確的程式，再按執行
recycle = pd.read_csv('臺北市各區資源回收量.csv', encoding='cp950')
recycle.columns = ['年度', '月', '行政區代碼', '區隊', '回收量(噸)']
r2025 = recycle[recycle['年度'] == 114].groupby('區隊', as_index=False)['回收量(噸)'].sum()
r2025['區域'] = '臺北市' + r2025['區隊'].str.replace(___, '區')          # 大安區隊 → 臺北市大安區

per_kg = r2025.merge(pop[pop['西元年'] == 2025], on=___, how='inner')
per_kg['每人回收(公斤)'] = (per_kg['回收量(噸)'] * 1000 / per_kg['人口']).round(1)
best_district = per_kg.loc[per_kg['每人回收(公斤)'].idxmax(), '區域']
display(per_kg[['區域', '回收量(噸)', '人口', '每人回收(公斤)']].sort_values('每人回收(公斤)', ascending=False))
print('每人回收最多：', best_district)

In [ ]:
# @title ✅ 執行這格，檢查練習 2
_r = pd.read_csv('臺北市各區資源回收量.csv', encoding='cp950')
_r.columns = ['年度', '月', '行政區代碼', '區隊', '回收量(噸)']
_r = _r[_r['年度'] == 114].groupby('區隊', as_index=False)['回收量(噸)'].sum()
_r['區域'] = '臺北市' + _r['區隊'].str.replace('區隊', '區')
_m = _r.merge(pop[pop['西元年'] == 2025], on='區域')
check('best_district', _m.loc[(_m['回收量(噸)'] / _m['人口']).idxmax(), '區域'])

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
recycle = pd.read_csv('臺北市各區資源回收量.csv', encoding='cp950')
recycle.columns = ['年度', '月', '行政區代碼', '區隊', '回收量(噸)']
r2025 = recycle[recycle['年度'] == 114].groupby('區隊', as_index=False)['回收量(噸)'].sum()
r2025['區域'] = '臺北市' + r2025['區隊'].str.replace('區隊', '區')

per_kg = r2025.merge(pop[pop['西元年'] == 2025], on='區域', how='inner')
per_kg['每人回收(公斤)'] = (per_kg['回收量(噸)'] * 1000 / per_kg['人口']).round(1)
best_district = per_kg.loc[per_kg['每人回收(公斤)'].idxmax(), '區域']
display(per_kg[['區域', '回收量(噸)', '人口', '每人回收(公斤)']].sort_values('每人回收(公斤)', ascending=False))
print('每人回收最多：', best_district)
# 「其他」區隊對不到任何行政區，inner 合併時會被排除——合併後記得檢查少了什麼
```

</details>

### ✍️ 練習 3：把縣市人口的寬表轉回長表

<details><summary>💡 卡住了？點我看提示</summary>

`county` 是「縣市 × 年」的寬表。先 `drop(columns='兩年變化%')`、`reset_index()`，再 `melt(id_vars='縣市', var_name='西元年', value_name='人口')`。

</details>

In [ ]:
# ✏️ 練習 3：把 ___ 換成正確的程式，再按執行
county_long = (county.drop(columns='兩年變化%').reset_index()
                     .melt(id_vars=___, var_name='西元年', value_name=___))
print(county_long.shape)
county_long.head()

In [ ]:
# @title ✅ 執行這格，檢查練習 3
check('county_long', county.drop(columns='兩年變化%').reset_index()
                     .melt(id_vars='縣市', var_name='西元年', value_name='人口'))

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
county_long = (county.drop(columns='兩年變化%').reset_index()
                     .melt(id_vars='縣市', var_name='西元年', value_name='人口'))
print(county_long.shape)
county_long.head()
```

</details>

---

## 🏆 任務完成！今天你用程式做到了這些事

- ✅ 任務 1：三年的人口檔，上下接成一張表——`concat`
- ✅ 任務 2：數字是文字、還有「…」——轉型態
- ✅ 任務 3：把用電和人口合併——`merge`，然後……對不上！
- ✅ 任務 4：寬表與長表——`pivot` 和 `melt`
- ✅ 任務 5：哪些縣市人口在增加？——長表轉寬表算成長率

> 🎯 **帶走一句話**：程式只是工具，真正有價值的是你從資料裡找到的那句「所以我們應該……」。
> 把上面每個 💡 **看懂結果** 用自己的話寫成一句建議，就是一份可以拿去報告的分析。